# VOC Dataset Visualization

This notebook loads and visualizes samples from the VOC Detection and VOC Classification datasets to verify that the data processing pipeline and transforms are working correctly.

## 1. Import Libraries and Setup
We import the necessary standard libraries and the custom `get_datasets` function. We also define `n_time_bins`, which is required by the dataset loader signature (primarily for neuromorphic data) but is unused for static image datasets like VOC.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import sys
import os

# Ensure the project root is in the path to import src
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.utils.data_utils import get_datasets

# Dummy value for static images, required by the function signature
n_time_bins = 1

## 2. Define Visualization Helper
The datasets apply standard ImageNet normalization. To visualize the images correctly, we need a helper function to un-normalize the tensors (reverse the mean and standard deviation) and convert them to a format compatible with matplotlib.

In [ ]:
def imshow(tensor, title=None):
    """
    Un-normalizes a tensor and displays it using matplotlib.
    Assumes ImageNet normalization mean and std.
    """
    # ImageNet mean and std
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    # Convert from (C, H, W) to (H, W, C)
    img = tensor.permute(1, 2, 0).numpy()
    
    # Un-normalize
    img = std * img + mean
    
    # Clip to valid range [0, 1] for display
    img = np.clip(img, 0, 1)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

## 3. Visualize VOC Detection Data
Here we load the `voc_detection` dataset. This dataset returns the image and a dictionary containing the raw annotation (bounding boxes, object names, etc.). We will iterate through a few samples to verify the images load and the annotations are accessible.

In [ ]:
# Load the VOC Detection dataset
print("Loading VOC Detection dataset...")
train_ds_det, _ = get_datasets("voc_detection", n_time_bins)

print(f"Dataset loaded. Number of samples: {len(train_ds_det)}")

# Visualize a few samples
num_samples = 3
for i in range(num_samples):
    img, target = train_ds_det[i]
    
    # The target is a dictionary containing 'annotation'
    # We print the objects found in the image
    objects = target['annotation']['object']
    if not isinstance(objects, list):
        objects = [objects]
    
    object_names = [obj['name'] for obj in objects]
    print(f"Sample {i+1}: Objects found -> {object_names}")
    print(f"Raw Annotation Keys: {target['annotation'].keys()}")
    
    imshow(img, title=f"VOC Detection Sample {i+1}")

## 4. Visualize VOC Classification Data
Next, we load the `voc_classification` dataset. This wrapper extracts a specific class label (integer) for the image. We verify that the label maps correctly to a class name using the `classes` list provided by the dataset object.

In [ ]:
print("Loading VOC Multi-label dataset...")
train_ds_voc, _ = get_datasets("voc_multilabel", n_time_bins)

print(f"Dataset loaded. Number of samples: {len(train_ds_voc)}")

# Visualize a few samples
num_samples = 3
for i in range(num_samples):
    img, target = train_ds_voc[i]
    
    # Target is a multi-hot vector. Find indices where value is 1.
    indices = torch.where(target == 1)[0]
    class_names = [train_ds_voc.classes[idx] for idx in indices]
    
    print(f"Sample {i+1}: Labels -> {class_names}")
    
    imshow(img, title=f"VOC Multi-label: {class_names}")

In [ ]:
print("\nLoading COCO Multi-label dataset...")
train_ds_coco, _ = get_datasets("coco_multilabel", n_time_bins)

print(f"Dataset loaded. Number of samples: {len(train_ds_coco)}")

# Visualize a few samples
num_samples = 3
for i in range(num_samples):
    img, target = train_ds_coco[i]
    
    # Target is a multi-hot vector
    indices = torch.where(target == 1)[0]
    class_names = [train_ds_coco.classes[idx] for idx in indices]
    
    print(f"Sample {i+1}: Labels -> {class_names}")
    
    imshow(img, title=f"COCO Multi-label: {class_names}")